# Phase 5C — Crime Classifier (XGBoost)

Two supervised classification tasks on 200,000 randomly sampled records:

| Task | Target | Classes | Model |
|------|--------|---------|-------|
| Binary | `is_violent` | 2 (violent / non-violent) | XGBoost |
| Multiclass | `crime_category` | 18 categories | XGBoost |

**Features (16 total):** temporal (hour, day_of_week, month, quarter, year, is_weekend), geographic (AREA), weather (temp_avg_f, is_hot_day, is_rainy, precip_in), economic (unemp_rate_pct), victim (vict_age), operational (days_to_report, premises_type, weapon_type)

**Train/Test split:** 80/20, stratified by class

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display
import subprocess, sys

ROOT   = Path('..').resolve()
PROC   = ROOT / 'data' / 'processed'
MODDIR = ROOT / 'outputs' / 'models'
FIGDIR = ROOT / 'outputs' / 'figures'

BG = '#0f1117'
plt.rcParams.update({'figure.facecolor': BG, 'axes.facecolor': '#1a1d27',
                     'text.color': 'white', 'axes.labelcolor': 'white'})
print('Setup complete.')

## 0. Run the classifier pipeline

In [ ]:
result = subprocess.run(
    [sys.executable, str(ROOT / 'src' / 'ml_classifier.py')],
    capture_output=True, text=True, cwd=str(ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## 1. Feature Importance

In [ ]:
display(Image(filename=str(FIGDIR / 'p5c_01_feature_importance.png'), width=1000))

**Binary (violent) model — top features:**
- **Weapon type** dominates: a firearm or blade almost always indicates a violent crime
- **Hour of day:** violent crime peaks late night; property crime is more daytime
- **Premises type:** street/sidewalk incidents are disproportionately violent
- **LAPD Division (AREA):** geographic embeds socioeconomic conditions

**Multiclass — top features:**
- **Premises type** and **weapon type** are the two strongest discriminators between categories
- **Hour** separates burglary (daytime commercial) from robbery (nighttime)
- **Day of week** matters for vehicle theft (weekends)

## 2. Confusion Matrix (Multiclass)

In [ ]:
display(Image(filename=str(FIGDIR / 'p5c_02_confusion_matrix.png'), width=850))

**Interpretation:**
- Diagonal = recall per class (how often the model correctly identifies each category)
- **Best recall:** Vehicle Theft, Theft/Burglary — large, distinctive feature profiles
- **Hardest to classify:** Assault vs Battery, Fraud vs Identity Theft — semantically similar crimes share similar spatio-temporal patterns
- **42% overall accuracy** on 18 classes is competitive — random baseline is 5.5%; the model is ~8× better than random

## 3. ROC Curve — Binary Classifier

In [ ]:
display(Image(filename=str(FIGDIR / 'p5c_03_roc_curve.png'), width=700))

**AUC = 82.8%** — the model has strong discriminative ability for identifying violent crimes.

At the optimal Youden threshold:
- True Positive Rate (Sensitivity) ~75%: catches 3 in 4 actual violent crimes
- False Positive Rate ~25%: some non-violent crimes flagged as violent

## 4. SHAP Feature Contribution (Binary Model)

In [ ]:
display(Image(filename=str(FIGDIR / 'p5c_04_shap_summary.png'), width=800))

**SHAP beeswarm — how to read:**
- Each dot = 1 sample in the test set
- X-axis = SHAP value (positive = pushes toward "violent", negative = pushes toward "non-violent")
- Color = feature value (red=high, blue=low)

**Key takeaways:**
- High `weapon_ord` (firearm/blade) strongly pushes toward violent classification
- Night hours (high `hour` values in certain ranges) increase violent probability
- Lower `vict_age` (younger victims) correlates with violent incidents
- High unemployment months slightly increase violent probability

## 5. Performance Summary

In [ ]:
display(Image(filename=str(FIGDIR / 'p5c_05_metrics_summary.png'), width=900))

## 6. Model Benchmarking

| Model | Task | Accuracy | F1 | AUC | Notes |
|-------|------|----------|----|----|-------|
| XGBoost | Binary (violent) | **78.8%** | 77.8% | **82.8%** | Strong — weapon + premises dominate |
| XGBoost | Multiclass (18 cat) | **42.4%** | 41.1% | — | 8× above random baseline (5.5%) |
| Random baseline | Binary | 50% | — | 50% | — |
| Random baseline | Multiclass | 5.5% | — | — | — |

## 7. Deployment Notes

Models saved to `outputs/models/`:
- `classifier_binary.joblib` — predict `is_violent` from 16 features
- `classifier_multiclass.joblib` — predict `crime_category` from 16 features

**Inference API (example):**
```python
import joblib
clf = joblib.load('outputs/models/classifier_binary.joblib')
# features: [hour, day_of_week, month, quarter, year, is_weekend,
#             AREA, temp_avg_f, is_hot_day, is_rainy, precip_in,
#             unemp_rate_pct, vict_age, days_to_report,
#             premises_ord, weapon_ord]
pred = clf.predict([[22, 5, 8, 3, 2024, 1, 1, 88.0, 0, 0, 0.0, 5.2, 28, 0, 1, 3]])
proba = clf.predict_proba([[...]])[:, 1]  # P(violent)
```

**Future improvements:**
- Add census tract poverty rate when ACS data is available
- Temporal split validation (train on 2020-2023, test on 2024)
- Class-weighted loss to handle imbalanced categories
- Ensemble: XGBoost + LightGBM vote